### IPhone sales analysis

In [21]:
# step1 - import required library
import pandas as pd        # load and manipulate the csv file
import numpy as np         # numeric operations pandas relies on
import plotly.express as px     
import plotly.graph_objects as go
import re
from pathlib import Path
 

import matplotlib.pyplot as plt
import seaborn as sns
 
sns.set_theme(style="whitegrid", palette="deep")

# Path to the folder containing the notebook/CSV
CHART_PATH = Path.cwd()
print(CHART_PATH)
print(list(CHART_PATH.iterdir()))

E:\neha work\Data Analytics\Projects\Iphone sales Analysis
[WindowsPath('E:/neha work/Data Analytics/Projects/Iphone sales Analysis/.ipynb_checkpoints'), WindowsPath('E:/neha work/Data Analytics/Projects/Iphone sales Analysis/apple_products.csv'), WindowsPath('E:/neha work/Data Analytics/Projects/Iphone sales Analysis/chart.ipynb'), WindowsPath('E:/neha work/Data Analytics/Projects/Iphone sales Analysis/claen and analyze data.ipynb'), WindowsPath('E:/neha work/Data Analytics/Projects/Iphone sales Analysis/model_summary.csv'), WindowsPath('E:/neha work/Data Analytics/Projects/Iphone sales Analysis/model_summary_csv')]


#### READ AND LOAD A DATA

In [7]:
# step2  -read a csv file with the pandas or load the csv file
df = pd.read_csv(CHART_PATH/"apple_products.csv")

In [8]:
# check the first 5 row of the file
df.head()

,index,Product Name,Product URL,Brand,Sale Price,Mrp,Discount Percentage,Number Of Ratings,Number Of Reviews,Upc,Star Rating,Ram
0,0,"APPLE iPhone 8 Plus (Gold, 64 GB)",https://www.flipkart.com/apple-iphone-8-plus-g...,Apple,49900,49900,0,3431,356,MOBEXRGV7EHHTGUH,4.6,2 GB
1,1,"APPLE iPhone 8 Plus (Space Grey, 256 GB)",https://www.flipkart.com/apple-iphone-8-plus-s...,Apple,84900,84900,0,3431,356,MOBEXRGVAC6TJT4F,4.6,2 GB
2,2,"APPLE iPhone 8 Plus (Silver, 256 GB)",https://www.flipkart.com/apple-iphone-8-plus-s...,Apple,84900,84900,0,3431,356,MOBEXRGVGETABXWZ,4.6,2 GB
3,3,"APPLE iPhone 8 (Silver, 256 GB)",https://www.flipkart.com/apple-iphone-8-silver...,Apple,77000,77000,0,11202,794,MOBEXRGVMZWUHCBA,4.5,2 GB
4,4,"APPLE iPhone 8 (Gold, 256 GB)",https://www.flipkart.com/apple-iphone-8-gold-2...,Apple,77000,77000,0,11202,794,MOBEXRGVPK7PFEJZ,4.5,2 GB


In [9]:
#step 3 - inspect the data
df.shape   # how many row and column is there

(62, 12)

In [10]:
#check dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   index                62 non-null     int64  
 1   Product Name         62 non-null     object 
 2   Product URL          62 non-null     object 
 3   Brand                62 non-null     object 
 4   Sale Price           62 non-null     int64  
 5   Mrp                  62 non-null     int64  
 6   Discount Percentage  62 non-null     int64  
 7   Number Of Ratings    62 non-null     int64  
 8   Number Of Reviews    62 non-null     int64  
 9   Upc                  62 non-null     object 
 10  Star Rating          62 non-null     float64
 11  Ram                  62 non-null     object 
dtypes: float64(1), int64(6), object(5)
memory usage: 5.9+ KB


#### STEP2 - CLEAN THE DATA

In [11]:
#step 2 - clean the data
# 1. Make a copy so the original data is preserved
df_clean = df.copy() 

#checking the missing value
print(df.isnull().sum())

index                  0
Product Name           0
Product URL            0
Brand                  0
Sale Price             0
Mrp                    0
Discount Percentage    0
Number Of Ratings      0
Number Of Reviews      0
Upc                    0
Star Rating            0
Ram                    0
dtype: int64


In [12]:
#states for numeric coulmns or check statistical values

df.describe()

,index,Sale Price,Mrp,Discount Percentage,Number Of Ratings,Number Of Reviews,Star Rating
count,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000,62.000000
mean,30.500000,80073.887097,88058.064516,9.951613,22420.403226,1861.677419,4.575806
std,18.041619,34310.446132,34728.825597,7.608079,33768.589550,2855.883830,0.059190
min,0.000000,29999.000000,39900.000000,0.000000,542.000000,42.000000,4.500000
25%,15.250000,49900.000000,54900.000000,6.000000,740.000000,64.000000,4.500000
50%,30.500000,75900.000000,79900.000000,10.000000,2101.000000,180.000000,4.600000
75%,45.750000,117100.000000,120950.000000,14.000000,43470.000000,3331.000000,4.600000
max,61.000000,140900.000000,149900.000000,29.000000,95909.000000,8161.000000,4.700000


In [13]:
# check duplicate rows
df.duplicated().sum()

np.int64(0)

In [14]:
print(df['Ram'].unique())

['2 GB' '4 GB' '3 GB' '6 GB']


In [15]:
#fix a messy column
#because Ram look like string not a number 
#why?- you can not calclate text 

df['Ram'] = df['Ram'].astype(str).str.extract(r'(/d+)').astype(float)

In [16]:
#view column names
df.columns

Index(['index', 'Product Name', 'Product URL', 'Brand', 'Sale Price', 'Mrp',
       'Discount Percentage', 'Number Of Ratings', 'Number Of Reviews', 'Upc',
       'Star Rating', 'Ram'],
      dtype='object')

### Exract model, color, storage from the product name

In [17]:
# product name column has mix type of data

import re   # re = regular expression" - to finding pattern in text

#create a fucntion
def parse_name(name):
    #clean up the text
    n= name.replace("APPLE ", "").replace("Apple ","")
    #find the model name and the bracket part
    m = re.match(r"(iPhone [\w\s]+?)\s*\((.+)\)", n)  #"find the words starting with 'iPhone', then find whatever is inside the round brackets right after it."
    #save the model
    model = m.group(1).strip() if m else n
    #split the bracket contents
    inside = m.group(2) if m else ""
    parts = [p.strip() for p in inside.split(",")]
    #pull out color
    color = parts[0] if len(parts) >0 else None 
    #find the strorage
    storage = None
    for p in parts:
        sm= re.search(r"(\d+)\s?GB", p)
        if sm:
            storage = sm.group(1) + " GB"
    return pd.Series([model, color, storage])  

df[["Model", "Color", "Storage"]] = df["Product Name"].apply(parse_name)


print(df[["Product Name", "Model", "Color", "Storage"]].head(10))

                                        Product Name          Model  \
0                  APPLE iPhone 8 Plus (Gold, 64 GB)  iPhone 8 Plus   
1           APPLE iPhone 8 Plus (Space Grey, 256 GB)  iPhone 8 Plus   
2               APPLE iPhone 8 Plus (Silver, 256 GB)  iPhone 8 Plus   
3                    APPLE iPhone 8 (Silver, 256 GB)       iPhone 8   
4                      APPLE iPhone 8 (Gold, 256 GB)       iPhone 8   
5                APPLE iPhone 8 Plus (Silver, 64 GB)  iPhone 8 Plus   
6            APPLE iPhone 8 Plus (Space Grey, 64 GB)  iPhone 8 Plus   
7                APPLE iPhone 8 (Space Grey, 256 GB)       iPhone 8   
8                APPLE iPhone XS Max (Silver, 64 GB)  iPhone XS Max   
9  Apple iPhone XR ((PRODUCT)RED, 128 GB) (Includ...      iPhone XR   

          Color Storage  
0          Gold   64 GB  
1    Space Grey  256 GB  
2        Silver  256 GB  
3        Silver  256 GB  
4          Gold  256 GB  
5        Silver   64 GB  
6    Space Grey   64 GB  
7    Space

### step 3 analyze  - Group by model

In [18]:
model_summary = df.groupby("Model").agg(
    listings=("Model","count"),
    avg_price=("Sale Price", "mean"),
    avg_discount_pct=("Number Of Ratings", "sum"),
    total_ratings=("Number Of Reviews", "sum"),
    avg_star=("Star Rating", "mean"),

).round(1).sort_values("total_ratings", ascending=False)

#convert into csv file
model_summary.to_csv("model_summary.csv")
print(model_summary)

                   listings  avg_price  avg_discount_pct  total_ratings  \
Model                                                                     
iPhone SE                 6    34999.0            575250          48952   
iPhone XR                 5    41599.0            397630          33988   
iPhone 11                 7    50427.6            304764          23369   
iPhone 8                  3    77000.0             33606           2382   
iPhone 11 Pro             4   102674.8             28345           2091   
iPhone 8 Plus             5    63900.0             17155           1780   
iPhone 12                 7    74471.4             14689           1256   
iPhone 11 Pro Max         5   123020.0              5390            505   
iPhone 12 Mini            6    62400.0              4420            382   
iPhone 12 Pro Max         8   125900.0              4640            360   
iPhone 12 Pro             5   124900.0              2722            210   
iPhone XS Max            

### price bands + correlation

In [19]:
#Price bands tell you where most of your listings sit
#Correlation tells you what actually drives popularity

#groupp price into ranges
#the price boundaries you're cutting at
bins = [0, 40000,60000,90000,130000,200000]
#name of the each range
labels =["<40K", "40-60k","60-90k","90-130k","130k+"]
#takes each phone's Sale Price and sorts it into one of those ranges, storing the result as a new column Price Band.
df["Price Band"] = pd.cut(df["Sale Price"],bins=bins, labels=labels)
print(df['Price Band'].value_counts().sort_index())

print("\n=== Avg discount by RAM tier ===")
print(df.groupby("Ram")["Discount Percentage"].mean().round(1))

# find which numbers move together
print("\n=== Correlation: Discount % vs Number of Ratings ===")
print(df[["Discount Percentage","Number Of Ratings","Sale Price","Star Rating"]].corr().round(2))

Price Band
<40K        6
40-60k     18
60-90k     17
90-130k    13
130k+       8
Name: count, dtype: int64

=== Avg discount by RAM tier ===
Series([], Name: Discount Percentage, dtype: float64)

=== Correlation: Discount % vs Number of Ratings ===
                     Discount Percentage  Number Of Ratings  Sale Price  \
Discount Percentage                 1.00               0.68       -0.57   
Number Of Ratings                   0.68               1.00       -0.70   
Sale Price                         -0.57              -0.70        1.00   
Star Rating                        -0.35              -0.22        0.30   

                     Star Rating  
Discount Percentage        -0.35  
Number Of Ratings          -0.22  
Sale Price                  0.30  
Star Rating                 1.00  


#### TOP 10 BEST SELLING IPHONES

In [20]:
#check the top 10 highest sales iphone by number of rating or stat rating
top_sales = df.sort_values(by="Number Of Ratings", ascending=False)
top_sales.head(10)

,index,Product Name,Product URL,Brand,Sale Price,Mrp,Discount Percentage,Number Of Ratings,Number Of Reviews,Upc,Star Rating,Ram,Model,Color,Storage,Price Band
23,23,"Apple iPhone SE (White, 256 GB) (Includes EarP...",https://www.flipkart.com/apple-iphone-se-white...,Apple,44999,54900,18,95909,8161,MOBFRFXHPZCHAPEH,4.5,NaN,iPhone SE,White,256 GB,40-60k
53,53,"APPLE iPhone SE (Black, 128 GB)",https://www.flipkart.com/apple-iphone-se-black...,Apple,34999,44900,22,95909,8161,MOBFWQ6BHUEVZPXD,4.5,NaN,iPhone SE,Black,128 GB,<40K
55,55,"APPLE iPhone SE (Red, 128 GB)",https://www.flipkart.com/apple-iphone-se-red-1...,Apple,34999,44900,22,95909,8161,MOBFWQ6BJTVFKPEJ,4.5,NaN,iPhone SE,Red,128 GB,<40K
57,57,"APPLE iPhone SE (Black, 64 GB)",https://www.flipkart.com/apple-iphone-se-black...,Apple,29999,39900,24,95909,8161,MOBFWQ6BR3MK7AUG,4.5,NaN,iPhone SE,Black,64 GB,<40K
52,52,"APPLE iPhone SE (White, 64 GB)",https://www.flipkart.com/apple-iphone-se-white...,Apple,29999,39900,24,95807,8154,MOBFWQ6BGWDVGF3E,4.5,NaN,iPhone SE,White,64 GB,<40K
54,54,"APPLE iPhone SE (White, 128 GB)",https://www.flipkart.com/apple-iphone-se-white...,Apple,34999,44900,22,95807,8154,MOBFWQ6BJEHMUUZY,4.5,NaN,iPhone SE,White,128 GB,<40K
11,11,"Apple iPhone XR (Coral, 128 GB) (Includes EarP...",https://www.flipkart.com/apple-iphone-xr-coral...,Apple,41999,52900,20,79582,6804,MOBF9Z7ZS6GF5UAP,4.6,NaN,iPhone XR,Coral,128 GB,40-60k
13,13,"Apple iPhone XR (White, 128 GB) (Includes EarP...",https://www.flipkart.com/apple-iphone-xr-white...,Apple,41999,52900,20,79512,6796,MOBF9Z7ZZY3HCDZZ,4.6,NaN,iPhone XR,White,128 GB,40-60k
12,12,"Apple iPhone XR (Black, 128 GB) (Includes EarP...",https://www.flipkart.com/apple-iphone-xr-black...,Apple,41999,52900,20,79512,6796,MOBF9Z7ZYWNFGZUC,4.6,NaN,iPhone XR,Black,128 GB,40-60k
9,9,"Apple iPhone XR ((PRODUCT)RED, 128 GB) (Includ...",https://www.flipkart.com/apple-iphone-xr-produ...,Apple,41999,52900,20,79512,6796,MOBF9Z7ZHQC23PWQ,4.6,NaN,iPhone XR,(PRODUCT)RED,128 GB,40-60k
